# Step 3 - Apply the model and build the BL dataset

Applies the final model (`model_MMS_THEMIS_Cluster.pkl`, trained on MMS1+THA+THB+C1) to the 8 satellites, selects the BL points (pred_BL > 0.5), finds the closest clean MSP and MSH points (cusps cut Bz>=-50 nT on both, Np>=3 on MSH), averages their features, merges the satellites and adds the propagated OMNI values.

Inputs: `/DATA/ghisalberti/Datasets/{mission}/{sat}/{sat}_interesting_for_BL.pkl` (step 1), the model files in `/home/ghisalberti/GradientBoosting/diagnostics/`, OMNI files in `/DATA/ghisalberti/Datasets/OMNI/OMNI/`.

Output: `{OUT}/BL_MMS1_THEMIS_Cluster_without_cusps_withoutNp<3_all_sat_1min_2025-05-06.pkl`.

Note: `date_model='2025-05-06'` is only a label in file names, and the 2025-05-06 model file is read only for the list of features used in `dropna`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # makes the blxline package importable
from blxline.paths import OUT, MODEL_LINES  # all outputs of this repo are written under OUT (never over the original files)

In [ ]:
# [from GradientBoosting/CommonPred.ipynb, cell 1]
import pandas as pd
from blxline.model import get_all_features_ensemble_model, pred_ensemble
import warnings
from datetime import datetime, timedelta
import numpy as np
import warnings
from blxline.dataframes import get_consecutive_interval
from spok.coordinates.coordinates import cartesian_to_spherical
from blxline.maps import associate_SW_Safrankova

In [ ]:
# [from GradientBoosting/CommonPred.ipynb, cell 2]
def get_common_features(features1,features2):
    common_features = []
    for col in features1:
        if col in features2:
            common_features += [col]
    return common_features

In [ ]:
# [from GradientBoosting/CommonPred.ipynb, cell 3]
#date_model = '2025-02-03'   # With searchcoil FFT
#date_model = '2025-02-25'  # Without searchcoil FFT
#date_model = '2025-03-27'  # Without searchcoil FFT
date_model = '2025-05-06'  # Without searchcoil FFT, and with tuned hyperparameters

nb_minutes_in_a_row = 1

without_coldions = False
without_cusps = True
densitycondition = True

all_sat = True

suffix = ''
if without_coldions :
    suffix += '_without_coldions'
if without_cusps:
    suffix += '_without_cusps'
if densitycondition:
    suffix += '_withoutNp<3'
if all_sat:
    suffix+='_all_sat'

## Predictions on the 8 satellites

In [ ]:
# [from GradientBoosting/CommonPred.ipynb, cell 6]
# Making the pred
warnings.filterwarnings("ignore")

model_results = pd.read_pickle(f'/home/ghisalberti/GradientBoosting/diagnostics/ensemble_model_results_{date_model}.pkl')
all_features = get_all_features_ensemble_model(model_results)

for sat, mission in zip(['MMS1','THA','THB','THC','THD','THE','C1','C3'], ['MMS']+['THEMIS']*5+['Cluster']*2):
    if all_sat:
        model_results = pd.read_pickle(f'/home/ghisalberti/GradientBoosting/diagnostics/model_MMS_THEMIS_Cluster.pkl')
    elif mission == 'Cluster':
        model_results = pd.read_pickle(f'/home/ghisalberti/GradientBoosting/diagnostics/ensemble_model_cluster_2025-08-05.pkl')
    else:
        model_results = pd.read_pickle(f'/home/ghisalberti/GradientBoosting/diagnostics/ensemble_model_results_{date_model}.pkl')
    #for sat, mission in zip(['C1','C3'], ['Cluster']):
    #for sat, mission in zip(['THA','THB','THC','THD','THE'], ['THEMIS' for i in range(5)]):
    #for sat, mission in zip(['C3'], ['Cluster']):
    data = pd.read_pickle(f'/DATA/ghisalberti/Datasets/{mission}/{sat}/{sat}_interesting_for_BL.pkl')
    #data = data.drop(columns = ['label_BL']).dropna()
    data = data[all_features].dropna()
    pred = pred_ensemble(data, model_results)
    pred = pd.DataFrame(pred, index = data.index.values, columns = ['pred_BL'])
    #pred.to_pickle(f'{OUT}/all_pred/{sat}_pred_common_model_{suffix}.pkl')
    pred.to_pickle(f'{OUT}/all_pred/{sat}_pred_common_model_{date_model}.pkl')
    print(f'Done for {sat}')

## BL points, closest clean MSP and MSH points

In [ ]:
# [from GradientBoosting/CommonPred.ipynb, cell 20]
# Finding closest MSP and MSH, and Transition parameter
warnings.filterwarnings("ignore")

#for sat, mission in zip(['THD'], ['THEMIS']):
#for sat, mission in zip(['THA','THB','THC','THD','THE'], ['THEMIS']*5):
for sat, mission in zip(['MMS1','THA','THB','THC','THD','THE','C1','C3'], ['MMS']+['THEMIS']*5+['Cluster']*2):
#for sat, mission in zip(['MMS1'], ['MMS']):
    data = pd.read_pickle(f'/DATA/ghisalberti/Datasets/{mission}/{sat}/{sat}_interesting_for_BL.pkl')
    if 'regions_pred' not in data.columns:
        data['regions_pred']=data.pred_regions.values
    pred = pd.read_pickle(f'{OUT}/all_pred/{sat}_pred_common_model_{date_model}.pkl')
    data.loc[pred.index.values, 'pred_BL'] = pred.values
    R, theta, phi = cartesian_to_spherical(data.X.values, data.Y.values, data.Z.values)
    data['R'] = R
    data['theta'] = theta
    data['phi'] = phi
    data['N/T'] = data.Np.values/data.Tp.values

    BL = data[data.pred_BL.values > 0.5]
    BL['MSP_NoverT'] = np.nan
    BL['MSH_NoverT'] = np.nan
    starts, stops = get_consecutive_interval(BL.index.values)
    dates = starts + (stops-starts)/2
    clean_MSP = data[np.logical_and(data.regions_pred.values==0, data.pred_BL.values<0.1)]
    clean_MSH = data[np.logical_and(data.regions_pred.values==1, data.pred_BL.values<0.1)]
    if without_coldions == True:
        clean_MSP = clean_MSP[clean_MSP.max_coldions.values<=0]
        #clean_MSH = clean_MSH[clean_MSH.max_coldions.values<=0]
    if without_cusps:
        clean_MSH = clean_MSH[clean_MSH.Bz.values>=-50]
        clean_MSP = clean_MSP[clean_MSP.Bz.values>=-50]
    if densitycondition : 
        clean_MSH = clean_MSH[clean_MSH.Np.values>=3]

    closest_MSP_dict = {}
    closest_MSH_dict = {}
    
    for i in range(len(dates)):
        date, start, stop = dates[i], starts[i], stops[i]
        
        clean_MSP['dt'] = abs(clean_MSP.index.values - date)/10**9
        two_hours = 2*60*60
        temp_MSP = clean_MSP[clean_MSP.dt.values < two_hours]  # Only keep what is closer than 2h away
        temp_MSP = temp_MSP.sort_values(by='dt')
        nb_points = int(nb_minutes_in_a_row*60/5)
        if len(temp_MSP) >= nb_points:
            temp_MSP = temp_MSP[:nb_points]
            closest_MSP_dict[date] = temp_MSP.index.values
            BL.loc[start:stop, 'MSP_NoverT'] = np.median(temp_MSP.Np.values/temp_MSP.Tp.values)
        else:
            closest_MSP_dict[date] = []
    
        clean_MSH['dt'] = abs(clean_MSH.index.values - date)/10**9
        two_hours = 2*60*60
        temp_MSH = clean_MSH[clean_MSH.dt.values < two_hours]  # Only keep what is closer than 2h away
        temp_MSH = temp_MSH.sort_values(by='dt')
        nb_points = int(nb_minutes_in_a_row*60/5)
        if len(temp_MSH) >= nb_points:
            temp_MSH = temp_MSH[:nb_points]
            closest_MSH_dict[date] = temp_MSH.index.values
            BL.loc[start:stop, 'MSH_NoverT'] = np.median(temp_MSH.Np.values/temp_MSH.Tp.values)
        else:
            closest_MSH_dict[date] = []
        
        if i%(len(dates)//50)==0:
            print(f'Interval {i} / {len(dates)} done for {sat}.')

    
    pd.to_pickle(closest_MSP_dict, f'{OUT}/all_pred/closest_MSP_{sat}_BL{suffix}_{nb_minutes_in_a_row}min_{date_model}.pkl')
    pd.to_pickle(closest_MSH_dict, f'{OUT}/all_pred/closest_MSH_{sat}_BL{suffix}_{nb_minutes_in_a_row}min_{date_model}.pkl')
    BL['normalized_logNoverT'] = (np.log(BL['N/T'].values) - np.log(BL['MSP_NoverT'].values))/(np.log(BL['MSH_NoverT'].values) - np.log(BL['MSP_NoverT'].values))   # Linear transformation? or linear transformation of log
    BL.to_pickle(f'{OUT}/all_pred/BL_{sat}_closestMSPandMSH{suffix}_{nb_minutes_in_a_row}min_{date_model}.pkl')
    BL.to_pickle(f'{OUT}/all_pred/BL_{sat}_closestMSPandMSH{suffix}_{nb_minutes_in_a_row}min_{date_model}_bis.pkl')
    print(f'{sat} is done!\n')


## Median of the neighbouring MSP and MSH points

In [ ]:
# [from GradientBoosting/CommonPred.ipynb, cell 25]
# Averaging neighbouring MSP and MSH points

satellites = ['MMS1','THA','THB','THC','THD','THE','C1','C3']
#satellites = ['THA','THD','THE']


closest_MSP_dates_list = {sat:pd.read_pickle(f'{OUT}/all_pred/closest_MSP_{sat}_BL{suffix}_{nb_minutes_in_a_row}min_{date_model}.pkl') for sat in satellites}
closest_MSH_dates_list = {sat:pd.read_pickle(f'{OUT}/all_pred/closest_MSH_{sat}_BL{suffix}_{nb_minutes_in_a_row}min_{date_model}.pkl') for sat in satellites}

paths = {#'MMS1':f"/home/ghisalberti/make_datasets/datasets/MMS1_dataframe_2024-12-04.pkl",
         'MMS1':f'/DATA/ghisalberti/Datasets/MMS/MMS1/MMS1_interesting_for_BL.pkl',
         'THA':f'/DATA/ghisalberti/Datasets/THEMIS/THA/THA_interesting_for_BL.pkl',
         'THB':f'/DATA/ghisalberti/Datasets/THEMIS/THB/THB_interesting_for_BL.pkl',
         'THC':f'/DATA/ghisalberti/Datasets/THEMIS/THC/THC_interesting_for_BL.pkl',
         'THD':f'/DATA/ghisalberti/Datasets/THEMIS/THD/THD_interesting_for_BL.pkl',
         'THE':f'/DATA/ghisalberti/Datasets/THEMIS/THE/THE_interesting_for_BL.pkl',
         'C1':f'/DATA/ghisalberti/Datasets/Cluster/C1/C1_interesting_for_BL.pkl',
         'C3':f'/DATA/ghisalberti/Datasets/Cluster/C3/C3_interesting_for_BL.pkl'}
datas = {sat:pd.read_pickle(paths[sat]) for sat in satellites}


warnings.filterwarnings("ignore")

#features = []
features = ['Btan1_MP','Btan2_MP','Bn_MP','Np','Vx','Vy','Vz','Vtan1_MP','Vtan2_MP', 'Vn_MP','Bx','By','Bz','Tp','Tpara','Tperp','energy_main_pop','flux_main_pop','width_main_pop']+[f'spectro_{i}' for i in range(32)]
new_features = [feat+'_MSP' for feat in features]+[feat+'_MSH' for feat in features]
MSP_features = ['max_coldions','center_coldions','std_coldions']
#MSP_features = []
MSP_new_features = [feat+'_MSP' for feat in MSP_features]

#for sat in ['MMS1','THA','THB','THC','THD','THE','C1','C3']:
for sat in satellites:
    temp = pd.read_pickle(f'{OUT}/all_pred/BL_{sat}_closestMSPandMSH{suffix}_{nb_minutes_in_a_row}min_{date_model}.pkl')

    for col in new_features:
        temp[col]= np.nan
    for col in MSP_new_features:
        temp[col]= np.nan
        
    closest_MSP_dates = closest_MSP_dates_list[sat]
    closest_MSH_dates = closest_MSH_dates_list[sat]
    data = datas[sat]
    starts_intervals, stops_intervals = get_consecutive_interval(temp.index.values)
    print(len(starts_intervals))

    count = 0
    for i, (start, stop) in enumerate(zip(starts_intervals, stops_intervals)):
        try:
            df_msp = data.loc[closest_MSP_dates[start + (stop-start)/2]]
            df_msh = data.loc[closest_MSH_dates[start + (stop-start)/2]]
            for col in features:
                temp.loc[start:stop, col+'_MSP'] = np.median(df_msp[col].values.flatten())
                temp.loc[start:stop, col+'_MSH'] = np.median(df_msh[col].values.flatten())
            for col in MSP_features:
                vals = df_msp[col].values.flatten()
                #print(vals)
                if (vals!=-1).sum() > 0:
                    vals[vals==-1]= np.nan
                temp.loc[start:stop, col+'_MSP'] = np.nanmedian(vals)
                
        except:
            count += 1
            
        if i%1000==0:
            print(f'{i} / {len(starts_intervals)} done!')
            
    temp.to_pickle(f'{OUT}/all_pred/BL_{sat}_closestMSPandMSH{suffix}_{nb_minutes_in_a_row}min_{date_model}.pkl')
    temp.to_pickle(f'{OUT}/all_pred/BL_{sat}_closestMSPandMSH{suffix}_{nb_minutes_in_a_row}min_{date_model}_bis.pkl')
    print(f'{sat} done! ({count} problems out of {len(temp)})\n')


## Merge the 8 satellites

In [ ]:
# [from GradientBoosting/CommonPred.ipynb, cell 41]
# Merging

BL = pd.DataFrame()
for sat in ['MMS1','THA','THB','THC','THD','THE','C1','C3']:
    temp = pd.read_pickle(f'{OUT}/all_pred/BL_{sat}_closestMSPandMSH{suffix}_{nb_minutes_in_a_row}min_{date_model}.pkl')
    temp['sat'] = sat
    if len(BL)>0:
        common_features = get_common_features(temp.columns, BL.columns)
        BL = BL[common_features]
        temp = temp[common_features]
    BL = pd.concat([BL, temp])
BL = BL.sort_index()

BL.to_pickle(f'{OUT}/BL_MMS1_THEMIS_Cluster{suffix}_{nb_minutes_in_a_row}min_{date_model}.pkl')
BL.to_pickle(f'{OUT}/BL_MMS1_THEMIS_Cluster_bis{suffix}_{nb_minutes_in_a_row}min_{date_model}.pkl')


## Add the propagated OMNI values

In [ ]:
# [from GradientBoosting/CommonPred.ipynb, cell 50]
# Add upstream OMNI values, propagated!
path = f'{OUT}/BL_MMS1_THEMIS_Cluster{suffix}_{nb_minutes_in_a_row}min_{date_model}'
BL = pd.read_pickle(path+'.pkl')

#omni = pd.read_pickle('/DATA/michotte/Old_Datasets/Datasets_These/OMNI/Datasets_OMNI/OMNI_data_5S_2000_2021.pkl')
#BS_standoff = pd.read_pickle('/DATA/michotte/Old_Datasets/Datasets_These/OMNI/Datasets_OMNI/OMNI_X_BS_5S_2000_2021.pkl')
omni = pd.read_pickle('/DATA/ghisalberti/Datasets/OMNI/OMNI/omni_data_5S_2000_2025.pkl')
BS_standoff = pd.read_pickle('/DATA/ghisalberti/Datasets/OMNI/OMNI/OMNI_X_BS_5S_2000_2025.pkl')
print(BS_standoff.index.values[0], BS_standoff.index.values[-1])
#BS_standoff = BS_standoff[BS_standoff.index.isin(BL.index)]
BL = BL[BL.index.isin(BS_standoff.index)]
BS_standoff = BS_standoff.loc[BL.index.values]
    
omni = associate_SW_Safrankova(BL[['X']], omni, BS_standoff)
del BS_standoff

for col in omni.columns:
    BL['omni_'+col] = omni[col].values
    
BL.to_pickle(path+'.pkl')
BL.to_pickle(path+'_bis.pkl')